# Weather Features Gold Export

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent.parent.resolve()
SILVER_WEATHER_PATH = PROJECT_ROOT / "Data" / "Silver_layer" / "Features" / "weather_features_hourly.csv"
GOLD_FEATURES_DIR = PROJECT_ROOT / "Data" / "Gold_layer" / "Features"
GOLD_WEATHER_PATH = GOLD_FEATURES_DIR / "weather_features_hourly_gold.csv"
AUDIT_PATH = GOLD_FEATURES_DIR / "weather_features_gold_drop_audit.csv"

GOLD_FEATURES_DIR.mkdir(parents=True, exist_ok=True)

DROP_COLUMNS = [
    "Is_Crosswind_15kt",
    "Is_Crosswind_20kt",
    "Is_Freezing",
    "Is_Gale_Wind",
    "Is_Strong_Wind",
    "Is_WMO_Fog_Code",
    "Is_WMO_Thunderstorm_Code",
    "Runway_Ice_Risk",
    "Is_Extreme_Heat",
]

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SILVER_WEATHER_PATH:", SILVER_WEATHER_PATH)
print("GOLD_WEATHER_PATH:", GOLD_WEATHER_PATH)
print("DROP_COLUMNS:", DROP_COLUMNS)


PROJECT_ROOT: /Users/nguyenhung/PycharmProjects/DS108_AeroDelay
SILVER_WEATHER_PATH: /Users/nguyenhung/PycharmProjects/DS108_AeroDelay/Data/Silver_layer/Features/weather_features_hourly.csv
GOLD_WEATHER_PATH: /Users/nguyenhung/PycharmProjects/DS108_AeroDelay/Data/Gold_layer/Features/weather_features_hourly_gold.csv
DROP_COLUMNS: ['Is_Crosswind_15kt', 'Is_Crosswind_20kt', 'Is_Freezing', 'Is_Gale_Wind', 'Is_Strong_Wind', 'Is_WMO_Fog_Code', 'Is_WMO_Thunderstorm_Code', 'Runway_Ice_Risk', 'Is_Extreme_Heat']


## 1. Load Silver weather features

In [2]:
if not SILVER_WEATHER_PATH.exists():
    raise FileNotFoundError(f"Missing Silver weather features: {SILVER_WEATHER_PATH}")

weather = pd.read_csv(SILVER_WEATHER_PATH, low_memory=False)
print("Silver weather shape:", weather.shape)
display(weather.head())
display(pd.DataFrame({"column": weather.columns}))


Silver weather shape: (6624, 63)


,time,Airport,temperature,precipitation,cloudcover,wind_speed,wind_direction,pressure,humidity,visibility,...,Is_Severe_Convection_Risk,Is_Freezing,Is_Extreme_Heat,Is_Thunderstorm_Risk,Convective_Severity_Score,Runway_Wet_Risk,Forced_Runway_Swap_Risk,Runway_Ice_Risk,Weather_Delay_Risk_Score,Aviation_Operational_Risk_Score
0,2025-12-15 00:00:00,DAD,22.9,0.2,33,15.8,38,1016.5,83,12400.0,...,0,0,0,0,0,1,0,0,0,1
1,2025-12-15 01:00:00,DAD,22.2,0.3,100,12.6,31,1016.4,87,10840.0,...,0,0,0,1,0,1,0,0,1,2
2,2025-12-15 02:00:00,DAD,22.0,0.3,100,14.2,33,1016.2,88,10760.0,...,0,0,0,1,0,1,0,0,1,2
3,2025-12-15 03:00:00,DAD,22.0,0.2,98,11.8,6,1015.7,88,10320.0,...,0,0,0,1,0,1,0,0,1,2
4,2025-12-15 04:00:00,DAD,22.0,0.1,100,9.4,1,1015.6,90,10380.0,...,0,0,0,1,0,1,0,0,1,2


,column
0,time
1,Airport
2,temperature
3,precipitation
4,cloudcover
...,...
58,Runway_Wet_Risk
59,Forced_Runway_Swap_Risk
60,Runway_Ice_Risk
61,Weather_Delay_Risk_Score


## 2. Drop selected columns and audit

In [3]:
drop_unique = list(dict.fromkeys(DROP_COLUMNS))
existing_drop_cols = [col for col in drop_unique if col in weather.columns]
missing_drop_cols = [col for col in drop_unique if col not in weather.columns]

weather_gold = weather.drop(columns=existing_drop_cols).copy()

audit = pd.DataFrame({
    "metric": [
        "silver_rows",
        "silver_columns",
        "gold_rows",
        "gold_columns",
        "dropped_columns_count",
        "missing_requested_drop_columns_count",
    ],
    "value": [
        len(weather),
        weather.shape[1],
        len(weather_gold),
        weather_gold.shape[1],
        len(existing_drop_cols),
        len(missing_drop_cols),
    ],
})

drop_detail = pd.DataFrame({
    "requested_drop_column": drop_unique,
    "was_present_in_silver": [col in existing_drop_cols for col in drop_unique],
})

print("Existing columns dropped:", existing_drop_cols)
print("Requested columns not found:", missing_drop_cols)
print("Gold weather shape:", weather_gold.shape)
display(audit)
display(drop_detail)


Existing columns dropped: ['Is_Crosswind_15kt', 'Is_Crosswind_20kt', 'Is_Freezing', 'Is_Gale_Wind', 'Is_Strong_Wind', 'Is_WMO_Fog_Code', 'Is_WMO_Thunderstorm_Code', 'Runway_Ice_Risk', 'Is_Extreme_Heat']
Requested columns not found: []
Gold weather shape: (6624, 54)


,metric,value
0,silver_rows,6624
1,silver_columns,63
2,gold_rows,6624
3,gold_columns,54
4,dropped_columns_count,9
5,missing_requested_drop_columns_count,0


,requested_drop_column,was_present_in_silver
0,Is_Crosswind_15kt,True
1,Is_Crosswind_20kt,True
2,Is_Freezing,True
3,Is_Gale_Wind,True
4,Is_Strong_Wind,True
5,Is_WMO_Fog_Code,True
6,Is_WMO_Thunderstorm_Code,True
7,Runway_Ice_Risk,True
8,Is_Extreme_Heat,True


## 3. Save Gold weather features

In [4]:
weather_gold.to_csv(GOLD_WEATHER_PATH, index=False, encoding="utf-8-sig")

# Save a compact audit table next to the output for reproducibility.
audit_out = pd.concat([
    audit.assign(section="summary"),
    drop_detail.rename(columns={"requested_drop_column": "metric", "was_present_in_silver": "value"}).assign(section="drop_detail"),
], ignore_index=True, sort=False)
audit_out.to_csv(AUDIT_PATH, index=False, encoding="utf-8-sig")

# Validate output headers.
written_cols = pd.read_csv(GOLD_WEATHER_PATH, nrows=0).columns.tolist()
remaining_drop_cols = [col for col in drop_unique if col in written_cols]
if remaining_drop_cols:
    raise AssertionError(f"Drop columns still present in Gold weather output: {remaining_drop_cols}")

print("Saved Gold weather features:", GOLD_WEATHER_PATH)
print("Saved drop audit:", AUDIT_PATH)
print("Remaining requested drop columns in output:", remaining_drop_cols)
print("Final shape:", (len(weather_gold), len(written_cols)))


Saved Gold weather features:

 /Users/nguyenhung/PycharmProjects/DS108_AeroDelay/Data/Gold_layer/Features/weather_features_hourly_gold.csv
Saved drop audit: /Users/nguyenhung/PycharmProjects/DS108_AeroDelay/Data/Gold_layer/Features/weather_features_gold_drop_audit.csv
Remaining requested drop columns in output: []
Final shape: (6624, 54)
